<a href="https://colab.research.google.com/github/syk-land/gitrepo/blob/main/vlm_%ED%98%B8%EC%B6%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 [Google Colab T4 GPU] 1024x1024 초고속 + OOM/터널끊김 완전 방지 서버

In [ ]:
# ============================================================
# [Google Colab T4 GPU] 1024x1024 일괄 생성 대응 + RAM 누수 방지 API 서버
# ============================================================
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

!pip uninstall -y torchao
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import gc, io, re, time, ctypes, threading, subprocess, traceback
import torch
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel
import uvicorn
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, EulerAncestralDiscreteScheduler

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

assert torch.cuda.is_available(), "❌ GPU가 꺼져 있습니다!"
print("✅ GPU 확인:", torch.cuda.get_device_name(0))

# 이미 VRAM에 모델이 로드되어 있으면 재로딩 건너뜀 (2중 적재 및 대기시간 방지)
if "pipe" not in globals():
    clean_memory()
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", dtype=torch.float16)
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        vae=vae,
        dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.load_lora_weights("nerijs/pixel-art-xl", weight_name="pixel-art-xl.safetensors")
    pipe.fuse_lora(lora_scale=1.0)

    # VRAM 직접 상주 + VAE 슬라이싱 & 타일링 (1024x1024 고속 생성 시 OOM 방지)
    pipe.to("cuda")
    pipe.enable_attention_slicing()
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()

    clean_memory()
    print("✅ 모델 로딩 및 VRAM 상주 완료!")
else:
    print("⚡ 이미 메모리에 로드된 모델을 재사용합니다!")

app = FastAPI()
gpu_lock = threading.Lock()
req_count = 0

class GenRequest(BaseModel):
    prompt: str
    negative_prompt: str = "3d, realistic, blurry, multiple characters, sprite sheet, grass, floor, shadow, text, watermark"
    width: int = 768
    height: int = 768
    steps: int = 20
    guidance_scale: float = 7.0
    seed: int | None = None

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0)}

@app.post("/generate")
def generate(req: GenRequest):
    global req_count
    try:
        with gpu_lock:
            generator = torch.Generator("cuda").manual_seed(req.seed) if req.seed is not None else None
            with torch.inference_mode():
                img = pipe(
                    prompt=req.prompt,
                    negative_prompt=req.negative_prompt,
                    width=req.width,
                    height=req.height,
                    num_inference_steps=req.steps,
                    guidance_scale=req.guidance_scale,
                    generator=generator,
                ).images[0]
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            img_bytes = buf.getvalue()
            buf.close()
            del img

            req_count += 1
            if req_count % 10 == 0:
                clean_memory()

            return Response(content=img_bytes, media_type="image/png")
    except Exception as e:
        clean_memory()
        return JSONResponse(status_code=500, content={"error": str(e), "trace": traceback.format_exc()})

if "server_started" not in globals():
    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
        daemon=True
    ).start()
    server_started = True
    time.sleep(2)

if "proc" in globals() and proc is not None:
    try:
        proc.terminate()
    except Exception:
        pass

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m and not public_url:
        public_url = m.group(0)
        print("\n" + "="*60)
        print(f"🌐 외부 API URL: {public_url}")
        print(f"   상태 확인: {public_url}/health")
        print(f"   이미지 생성: POST {public_url}/generate")
        print("="*60 + "\n")
        break

threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    proc.terminate()

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
✅ GPU 확인: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


... (17 outputs left out)

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]